# 05 — Twitch Candidate Generation v2

This notebook rebuilds the retrieval stage from the **raw Twitch interaction history**.

The previous version exposed two important limitations:

- only a small fraction of evaluation users had usable history;
- newly generated `(user, streamer)` pairs usually did not already exist in the prepared Notebook 02 feature table.

This version fixes the retrieval side by building directly from the raw temporal interaction history.

## Retrieval architecture

```text
historical Twitch interactions
          ↓
 ┌────────┼─────────┐
 ↓        ↓         ↓
History   SVD       Popularity
repeat    collaborative fallback
items     retrieval
 └────────┼─────────┘
          ↓
 Reciprocal-rank fusion
          ↓
 ~200 candidate streamers
          ↓
 learned ranker (later)
```

The notebook evaluates:

- Candidate Recall@50 / @100 / @200
- catalog coverage
- repeat-streamer recall
- new-streamer recall
- candidate-source contribution

It also saves **historical user, streamer, pair, and candidate base features** so arbitrary generated candidate pairs no longer depend on already existing in Notebook 02.


In [1]:
from __future__ import annotations

import gc
import json
import math
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

SEED = 42
rng = np.random.default_rng(SEED)

# ---------------------------------------------------------------------
# Locate project root robustly.
# ---------------------------------------------------------------------
cwd = Path.cwd()

PROJECT_CANDIDATES = [
    Path.home() / "Desktop" / "resume_projects" / "Distributed ML Inference & Ranking",
    cwd,
    *cwd.parents,
]

PROJECT_DIR = next(
    (
        p for p in PROJECT_CANDIDATES
        if (p / "data/raw/twitch/100k_a.csv").exists()
    ),
    None,
)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        "Could not find data/raw/twitch/100k_a.csv. "
        "Set PROJECT_DIR manually in this cell."
    )

RAW_PATH = PROJECT_DIR / "data/raw/twitch/100k_a.csv"
DATA_DIR = PROJECT_DIR / "data/processed/twitch"
ARTIFACT_DIR = PROJECT_DIR / "artifacts/twitch"
RESULTS_DIR = PROJECT_DIR / "results/05_twitch_candidate_generation"

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_PATH)
print("Results:", RESULTS_DIR)


Project: /Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking
Raw data: /Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/data/raw/twitch/100k_a.csv
Results: /Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/results/05_twitch_candidate_generation


## 1. Configuration

These defaults are intentionally laptop-friendly while still producing a meaningful retrieval system.

`MAX_SVD_USERS` and `MAX_SVD_STREAMERS` control only the latent-factor retrieval matrix. Raw historical statistics are still computed from the complete historical portion of the dataset.


In [2]:
RAW_COLUMNS = [
    "user_id",
    "stream_id",
    "streamer",
    "start_time",
    "stop_time",
]

CSV_CHUNK_SIZE = 200_000

# Temporal split: first 85% of event time = history, final 15% = future.
HISTORY_QUANTILE = 0.85

# Evaluation
MAX_EVAL_USERS = 2_000

# Latent-factor retriever
MAX_SVD_USERS = 50_000
MAX_SVD_STREAMERS = 60_000
N_COMPONENTS = 32
MIN_STREAMER_INTERACTIONS = 2

# Candidate-source sizes before fusion
HISTORY_SOURCE_K = 60
SVD_SOURCE_K = 300
POPULARITY_SOURCE_K = 200

# Final candidate pool
MAX_CANDIDATES = 200
RECALL_KS = [50, 100, 200]

# Reciprocal-rank-fusion settings
RRF_K = 60
SOURCE_WEIGHTS = {
    "history": 1.00,
    "svd": 1.00,
    "popularity": 0.55,
}

print("Max evaluation users:", MAX_EVAL_USERS)
print("Max SVD users:", f"{MAX_SVD_USERS:,}")
print("Max SVD streamers:", f"{MAX_SVD_STREAMERS:,}")
print("Max candidates/user:", MAX_CANDIDATES)


Max evaluation users: 2000
Max SVD users: 50,000
Max SVD streamers: 60,000
Max candidates/user: 200


## 2. Determine the temporal cutoff

We use the raw event time so candidate generation respects chronology.

No future interaction is used to build the retrieval model.


In [3]:
time_values = pd.read_csv(
    RAW_PATH,
    header=None,
    names=RAW_COLUMNS,
    usecols=["start_time"],
    dtype={"start_time": "int32"},
)["start_time"]

cutoff_time = int(
    np.floor(
        time_values.quantile(HISTORY_QUANTILE)
    )
)

print("Raw rows:", f"{len(time_values):,}")
print("Start time min:", int(time_values.min()))
print("Start time max:", int(time_values.max()))
print("History cutoff:", cutoff_time)

del time_values
gc.collect()


Raw rows: 3,051,733
Start time min: 0
Start time max: 6147
History cutoff: 5288


0

## 3. Aggregate historical and future interactions

Rather than loading the entire raw dataset and keeping every event in memory, we aggregate in chunks.

Historical tables contain:

- user-streamer interaction count
- total watch minutes
- last historical interaction time

The future table is used only as held-out truth.


In [4]:
history_parts = []
future_parts = []

rows_scanned = 0
history_events = 0
future_events = 0

start = perf_counter()

dtype_map = {
    "user_id": "int32",
    "stream_id": "int32",
    "streamer": "string",
    "start_time": "int32",
    "stop_time": "int32",
}

for chunk in pd.read_csv(
    RAW_PATH,
    header=None,
    names=RAW_COLUMNS,
    dtype=dtype_map,
    chunksize=CSV_CHUNK_SIZE,
):
    rows_scanned += len(chunk)

    chunk["watch_minutes"] = (
        chunk["stop_time"] - chunk["start_time"]
    ).astype("float32") * 10.0

    chunk = chunk[
        chunk["watch_minutes"] > 0
    ].copy()

    hist = chunk[
        chunk["start_time"] < cutoff_time
    ]

    fut = chunk[
        chunk["start_time"] >= cutoff_time
    ]

    history_events += len(hist)
    future_events += len(fut)

    if not hist.empty:
        history_parts.append(
            hist.groupby(
                ["user_id", "streamer"],
                as_index=False,
            ).agg(
                interactions=("stream_id", "size"),
                watch_minutes=("watch_minutes", "sum"),
                last_time=("start_time", "max"),
            )
        )

    if not fut.empty:
        future_parts.append(
            fut.groupby(
                ["user_id", "streamer"],
                as_index=False,
            ).agg(
                future_interactions=("stream_id", "size"),
                future_watch_minutes=("watch_minutes", "sum"),
            )
        )

    del chunk, hist, fut
    gc.collect()

history_pairs = pd.concat(
    history_parts,
    ignore_index=True,
)

history_pairs = (
    history_pairs.groupby(
        ["user_id", "streamer"],
        as_index=False,
    )
    .agg(
        interactions=("interactions", "sum"),
        watch_minutes=("watch_minutes", "sum"),
        last_time=("last_time", "max"),
    )
)

future_pairs = pd.concat(
    future_parts,
    ignore_index=True,
)

future_pairs = (
    future_pairs.groupby(
        ["user_id", "streamer"],
        as_index=False,
    )
    .agg(
        future_interactions=("future_interactions", "sum"),
        future_watch_minutes=("future_watch_minutes", "sum"),
    )
)

del history_parts, future_parts
gc.collect()

print("Rows scanned:", f"{rows_scanned:,}")
print("History events:", f"{history_events:,}")
print("Future events:", f"{future_events:,}")
print("Historical user-streamer pairs:", f"{len(history_pairs):,}")
print("Future user-streamer pairs:", f"{len(future_pairs):,}")
print("Historical users:", f"{history_pairs['user_id'].nunique():,}")
print("Historical streamers:", f"{history_pairs['streamer'].nunique():,}")
print("Aggregation seconds:", round(perf_counter() - start, 2))


Rows scanned: 3,051,733
History events: 2,593,535
Future events: 458,198
Historical user-streamer pairs: 1,337,200
Future user-streamer pairs: 359,679
Historical users: 98,802
Historical streamers: 148,585
Aggregation seconds: 2.17


## 4. Build historical user / streamer statistics

These tables are useful for both retrieval and later dynamic feature construction.


In [5]:
user_stats = (
    history_pairs.groupby("user_id")
    .agg(
        user_history_interactions=("interactions", "sum"),
        user_history_watch_minutes=("watch_minutes", "sum"),
        user_history_unique_streamers=("streamer", "nunique"),
        user_last_history_time=("last_time", "max"),
    )
    .reset_index()
)

streamer_stats = (
    history_pairs.groupby("streamer")
    .agg(
        streamer_history_interactions=("interactions", "sum"),
        streamer_history_watch_minutes=("watch_minutes", "sum"),
        streamer_history_unique_viewers=("user_id", "nunique"),
    )
    .reset_index()
)

streamer_stats["streamer_avg_watch_minutes"] = (
    streamer_stats["streamer_history_watch_minutes"]
    / streamer_stats["streamer_history_interactions"].clip(lower=1)
)

streamer_stats = streamer_stats.sort_values(
    "streamer_history_interactions",
    ascending=False,
).reset_index(drop=True)

print("User-stat rows:", f"{len(user_stats):,}")
print("Streamer-stat rows:", f"{len(streamer_stats):,}")

display(streamer_stats.head(10))


User-stat rows: 98,802
Streamer-stat rows: 148,585


,streamer,streamer_history_interactions,streamer_history_watch_minutes,streamer_history_unique_viewers,streamer_avg_watch_minutes
0,ninja,38701,968510.0,15581,25.025452
1,tfue,33636,883440.0,13261,26.264716
2,shroud,22705,603560.0,9906,26.582691
3,riotgames,15344,610970.0,7160,39.818170
4,dakotaz,12419,382720.0,5540,30.817296
5,esl_csgo,12151,383510.0,5042,31.562011
6,sodapoppin,11529,406040.0,5441,35.219013
7,asmongold,11282,447320.0,4409,39.648998
8,summit1g,10951,349150.0,4704,31.882933
9,nickmercs,10802,304110.0,4887,28.153120


## 5. Choose evaluation users

A candidate generator should be evaluated only for users who have both:

- historical behavior available at request time, and
- future interactions to act as held-out truth.

When Notebook 04's saved evaluation-user list exists, we keep as many of those users as possible for continuity, then fill the remaining slots from other eligible users.


In [6]:
history_users = set(
    history_pairs["user_id"].unique().tolist()
)

future_users = set(
    future_pairs["user_id"].unique().tolist()
)

eligible_users = np.array(
    sorted(history_users.intersection(future_users))
)

preferred_path = (
    ARTIFACT_DIR / "neural_eval_user_ids.csv"
)

preferred = []

if preferred_path.exists():
    preferred_raw = (
        pd.read_csv(preferred_path)["user_id"]
        .drop_duplicates()
        .tolist()
    )

    preferred = [
        int(u)
        for u in preferred_raw
        if int(u) in history_users
        and int(u) in future_users
    ]

preferred = list(dict.fromkeys(preferred))

remaining_needed = max(
    0,
    MAX_EVAL_USERS - len(preferred),
)

remaining_pool = np.array(
    [
        u for u in eligible_users
        if u not in set(preferred)
    ]
)

if remaining_needed > 0 and len(remaining_pool) > 0:
    extra_n = min(
        remaining_needed,
        len(remaining_pool),
    )

    extra = rng.choice(
        remaining_pool,
        size=extra_n,
        replace=False,
    ).tolist()
else:
    extra = []

eval_users = (
    preferred[:MAX_EVAL_USERS]
    + extra
)[:MAX_EVAL_USERS]

eval_user_set = set(eval_users)

print("Eligible history + future users:", f"{len(eligible_users):,}")
print("Notebook 04 users retained:", f"{min(len(preferred), MAX_EVAL_USERS):,}")
print("Final evaluation users:", f"{len(eval_users):,}")


Eligible history + future users: 71,164
Notebook 04 users retained: 0
Final evaluation users: 2,000


## 6. Build the SVD implicit-feedback matrix

The matrix uses historical interactions only.

The confidence-style weight combines frequency and watch time:

```text
log(1 + interactions) + 0.25 × log(1 + watch_minutes)
```

To keep the notebook Mac-friendly, the SVD stage keeps the most active users and streamers, while always retaining evaluation users whenever possible.


In [7]:
# ------------------------------------------------------------
# Select users for latent retrieval
# ------------------------------------------------------------
user_activity = (
    history_pairs.groupby("user_id")["interactions"]
    .sum()
    .sort_values(ascending=False)
)

top_users = user_activity.head(
    MAX_SVD_USERS
).index.tolist()

svd_user_ids = list(
    dict.fromkeys(
        eval_users + top_users
    )
)[:MAX_SVD_USERS]

svd_user_set = set(svd_user_ids)

# ------------------------------------------------------------
# Select streamers for latent retrieval
# ------------------------------------------------------------
streamer_activity = (
    streamer_stats.set_index("streamer")[
        "streamer_history_interactions"
    ]
)

eligible_streamers = streamer_activity[
    streamer_activity >= MIN_STREAMER_INTERACTIONS
].sort_values(ascending=False)

svd_streamers = (
    eligible_streamers
    .head(MAX_SVD_STREAMERS)
    .index
    .tolist()
)

svd_streamer_set = set(svd_streamers)

matrix_pairs = history_pairs[
    history_pairs["user_id"].isin(svd_user_set)
    & history_pairs["streamer"].isin(svd_streamer_set)
].copy()

user_to_idx = {
    user_id: idx
    for idx, user_id in enumerate(svd_user_ids)
}

streamer_to_idx = {
    streamer: idx
    for idx, streamer in enumerate(svd_streamers)
}

matrix_pairs["weight"] = (
    np.log1p(matrix_pairs["interactions"].astype("float32"))
    + 0.25
    * np.log1p(matrix_pairs["watch_minutes"].astype("float32"))
)

rows = matrix_pairs["user_id"].map(
    user_to_idx
).to_numpy()

cols = matrix_pairs["streamer"].map(
    streamer_to_idx
).to_numpy()

values = matrix_pairs["weight"].to_numpy(
    dtype=np.float32
)

R = csr_matrix(
    (values, (rows, cols)),
    shape=(
        len(svd_user_ids),
        len(svd_streamers),
    ),
    dtype=np.float32,
)

print("SVD users:", f"{len(svd_user_ids):,}")
print("SVD streamers:", f"{len(svd_streamers):,}")
print("Sparse matrix:", R.shape)
print("Non-zero pairs:", f"{R.nnz:,}")
print(
    "Density:",
    f"{R.nnz / (R.shape[0] * R.shape[1]):.6%}",
)


SVD users: 50,000
SVD streamers: 60,000
Sparse matrix: (50000, 60000)
Non-zero pairs: 999,082
Density: 0.033303%


## 7. Train the latent-factor retriever


In [8]:
n_components = min(
    N_COMPONENTS,
    max(2, min(R.shape) - 1),
)

svd = TruncatedSVD(
    n_components=n_components,
    n_iter=7,
    random_state=SEED,
)

start = perf_counter()

user_factors = svd.fit_transform(
    R
).astype(np.float32)

item_factors = (
    svd.components_.T
    .astype(np.float32)
)

elapsed = perf_counter() - start

print("SVD components:", n_components)
print("User factors:", user_factors.shape)
print("Item factors:", item_factors.shape)
print(
    "Explained variance:",
    f"{svd.explained_variance_ratio_.sum():.2%}",
)
print("Training seconds:", round(elapsed, 2))


SVD components: 32
User factors: (50000, 32)
Item factors: (60000, 32)
Explained variance: 20.75%
Training seconds: 0.46


## 8. Build retrieval lookup tables


In [9]:
# Strongest history items for each evaluation user.
eval_history = history_pairs[
    history_pairs["user_id"].isin(eval_user_set)
].copy()

eval_history["history_strength"] = (
    np.log1p(eval_history["interactions"])
    + 0.25 * np.log1p(eval_history["watch_minutes"])
)

eval_history = eval_history.sort_values(
    ["user_id", "history_strength"],
    ascending=[True, False],
)

history_lookup = {
    user_id: list(
        zip(
            group["streamer"].tolist(),
            group["history_strength"].tolist(),
        )
    )
    for user_id, group
    in eval_history.groupby("user_id", sort=False)
}

popular_streamers = (
    streamer_stats["streamer"]
    .tolist()
)

print(
    "Evaluation users with raw history:",
    f"{len(history_lookup):,} / {len(eval_users):,}",
)
print(
    "Popularity catalog size:",
    f"{len(popular_streamers):,}",
)


Evaluation users with raw history: 2,000 / 2,000
Popularity catalog size: 148,585


## 9. Retrieval-source functions


In [10]:
def retrieve_history(
    user_id,
    top_k=HISTORY_SOURCE_K,
):
    items = history_lookup.get(
        user_id,
        [],
    )[:top_k]

    return [
        (streamer, float(score))
        for streamer, score in items
    ]


def retrieve_svd(
    user_id,
    top_k=SVD_SOURCE_K,
):
    if user_id not in user_to_idx:
        return []

    uidx = user_to_idx[user_id]

    scores = (
        user_factors[uidx]
        @ item_factors.T
    )

    k = min(
        top_k,
        len(scores),
    )

    if k <= 0:
        return []

    top_idx = np.argpartition(
        scores,
        -k,
    )[-k:]

    top_idx = top_idx[
        np.argsort(
            scores[top_idx]
        )[::-1]
    ]

    return [
        (
            svd_streamers[idx],
            float(scores[idx]),
        )
        for idx in top_idx
    ]


def retrieve_popularity(
    top_k=POPULARITY_SOURCE_K,
):
    return [
        (
            streamer,
            float(1.0 / math.log2(rank + 2)),
        )
        for rank, streamer in enumerate(
            popular_streamers[:top_k],
            start=1,
        )
    ]


print("Retrieval functions ready.")


Retrieval functions ready.


## 10. Hybrid candidate generator

Raw SVD scores, history strengths, and popularity scores are on different scales.

Instead of adding incompatible raw scores, we use **weighted reciprocal-rank fusion**. Each source contributes according to the item's position within that source.


In [11]:
def generate_candidates(
    user_id,
    max_candidates=MAX_CANDIDATES,
):
    sources = {
        "history": retrieve_history(
            user_id,
            HISTORY_SOURCE_K,
        ),
        "svd": retrieve_svd(
            user_id,
            SVD_SOURCE_K,
        ),
        "popularity": retrieve_popularity(
            POPULARITY_SOURCE_K,
        ),
    }

    candidate_info = {}

    for source_name, items in sources.items():
        source_weight = SOURCE_WEIGHTS[source_name]

        for rank, (streamer, raw_score) in enumerate(
            items,
            start=1,
        ):
            row = candidate_info.setdefault(
                streamer,
                {
                    "fusion_score": 0.0,
                    "sources": set(),
                    "best_source_rank": {},
                },
            )

            row["fusion_score"] += (
                source_weight
                / (RRF_K + rank)
            )

            row["sources"].add(
                source_name
            )

            row["best_source_rank"][
                source_name
            ] = rank

    ranked = sorted(
        candidate_info.items(),
        key=lambda x: x[1]["fusion_score"],
        reverse=True,
    )[:max_candidates]

    output = []

    for final_rank, (streamer, meta) in enumerate(
        ranked,
        start=1,
    ):
        output.append(
            {
                "user_id": user_id,
                "streamer": streamer,
                "retrieval_rank": final_rank,
                "retrieval_score": meta["fusion_score"],
                "candidate_sources": "|".join(
                    sorted(meta["sources"])
                ),
                "history_rank": meta["best_source_rank"].get(
                    "history",
                    np.nan,
                ),
                "svd_rank": meta["best_source_rank"].get(
                    "svd",
                    np.nan,
                ),
                "popularity_rank": meta["best_source_rank"].get(
                    "popularity",
                    np.nan,
                ),
            }
        )

    return pd.DataFrame(output)


example_user = eval_users[0]

example_candidates = generate_candidates(
    example_user
)

print("Example user:", example_user)
print("Candidates:", len(example_candidates))
display(example_candidates.head(20))


Example user: 27441
Candidates: 200


,user_id,streamer,retrieval_rank,retrieval_score,candidate_sources,history_rank,svd_rank,popularity_rank
0,27441,elded,1,0.029350,history|popularity|svd,1.0,40.0,126.0
1,27441,tfue,2,0.025264,popularity|svd,NaN,1.0,2.0
2,27441,ninja,3,0.025145,popularity|svd,NaN,2.0,1.0
3,27441,elmiillor,4,0.019052,popularity|svd,NaN,3.0,113.0
4,27441,lvpes,5,0.018195,popularity|svd,NaN,4.0,154.0
5,27441,rubius,6,0.018015,popularity|svd,NaN,7.0,118.0
6,27441,chocotaco,7,0.017003,popularity|svd,NaN,33.0,28.0
7,27441,alexelcapo,8,0.016937,popularity|svd,NaN,9.0,165.0
8,27441,lirik,9,0.016679,popularity|svd,NaN,42.0,20.0
9,27441,izakooo,10,0.015932,popularity|svd,NaN,19.0,108.0


## 11. Generate candidate pools for evaluation users


In [12]:
candidate_frames = []

start = perf_counter()

for i, user_id in enumerate(
    eval_users,
    start=1,
):
    frame = generate_candidates(
        user_id,
        MAX_CANDIDATES,
    )

    if not frame.empty:
        candidate_frames.append(
            frame
        )

    if i % 250 == 0:
        print(
            f"Generated candidates for {i:,}/{len(eval_users):,} users"
        )

candidates = pd.concat(
    candidate_frames,
    ignore_index=True,
)

generation_seconds = (
    perf_counter() - start
)

candidate_counts = (
    candidates.groupby("user_id")
    .size()
)

print()
print("Candidate rows:", f"{len(candidates):,}")
print(
    "Users with candidates:",
    f"{candidates['user_id'].nunique():,}",
)
print(
    "Average candidates/user:",
    round(candidate_counts.mean(), 2),
)
print(
    "Median candidates/user:",
    round(candidate_counts.median(), 2),
)
print(
    "Generation seconds:",
    round(generation_seconds, 2),
)


Generated candidates for 250/2,000 users
Generated candidates for 500/2,000 users
Generated candidates for 750/2,000 users
Generated candidates for 1,000/2,000 users
Generated candidates for 1,250/2,000 users
Generated candidates for 1,500/2,000 users
Generated candidates for 1,750/2,000 users
Generated candidates for 2,000/2,000 users

Candidate rows: 400,000
Users with candidates: 2,000
Average candidates/user: 200.0
Median candidates/user: 200.0
Generation seconds: 2.99


## 12. Build held-out truth and measure catalog coverage

Some future streamers may never have appeared in historical data.

No historical recommender can retrieve those truly cold-start items, so we report:

- **raw future recall** — includes all future-positive streamers;
- **catalog-adjusted recall** — evaluates only future streamers that existed in the historical catalog;
- **catalog coverage** — how much of the future truth is theoretically retrievable.


In [13]:
future_eval = future_pairs[
    future_pairs["user_id"].isin(
        eval_user_set
    )
].copy()

historical_catalog = set(
    history_pairs["streamer"]
    .unique()
    .tolist()
)

future_truth = (
    future_eval.groupby("user_id")[
        "streamer"
    ]
    .agg(lambda s: set(s.tolist()))
    .to_dict()
)

catalog_truth = {
    user_id: {
        streamer
        for streamer in streamers
        if streamer in historical_catalog
    }
    for user_id, streamers
    in future_truth.items()
}

seen_lookup = {
    user_id: set(
        group["streamer"].tolist()
    )
    for user_id, group
    in eval_history.groupby(
        "user_id",
        sort=False,
    )
}

raw_truth_count = sum(
    len(items)
    for items in future_truth.values()
)

catalog_truth_count = sum(
    len(items)
    for items in catalog_truth.values()
)

catalog_coverage = (
    catalog_truth_count
    / raw_truth_count
    if raw_truth_count
    else np.nan
)

print(
    "Future-positive user-streamer pairs:",
    f"{raw_truth_count:,}",
)
print(
    "Future positives present in historical catalog:",
    f"{catalog_truth_count:,}",
)
print(
    "Historical-catalog coverage:",
    f"{catalog_coverage:.2%}",
)


Future-positive user-streamer pairs: 9,548
Future positives present in historical catalog: 9,080
Historical-catalog coverage: 95.10%


## 13. Candidate Recall@K — overall, repeat, and new streamers


In [14]:
candidate_lists = (
    candidates
    .sort_values(
        ["user_id", "retrieval_rank"]
    )
    .groupby("user_id")["streamer"]
    .agg(list)
    .to_dict()
)

metric_rows = []

for user_id in eval_users:
    retrieved = candidate_lists.get(
        user_id,
        [],
    )

    raw_truth = future_truth.get(
        user_id,
        set(),
    )

    retrievable_truth = catalog_truth.get(
        user_id,
        set(),
    )

    seen = seen_lookup.get(
        user_id,
        set(),
    )

    repeat_truth = (
        retrievable_truth
        .intersection(seen)
    )

    new_truth = (
        retrievable_truth
        .difference(seen)
    )

    if not raw_truth:
        continue

    row = {
        "user_id": user_id,
        "future_positive_count": len(raw_truth),
        "catalog_positive_count": len(retrievable_truth),
        "repeat_positive_count": len(repeat_truth),
        "new_positive_count": len(new_truth),
        "candidate_count": len(retrieved),
    }

    for k in RECALL_KS:
        topk = set(
            retrieved[:k]
        )

        raw_hits = len(
            raw_truth.intersection(topk)
        )

        catalog_hits = len(
            retrievable_truth.intersection(
                topk
            )
        )

        repeat_hits = len(
            repeat_truth.intersection(
                topk
            )
        )

        new_hits = len(
            new_truth.intersection(
                topk
            )
        )

        row[f"raw_recall@{k}"] = (
            raw_hits / len(raw_truth)
            if raw_truth
            else np.nan
        )

        row[f"catalog_recall@{k}"] = (
            catalog_hits
            / len(retrievable_truth)
            if retrievable_truth
            else np.nan
        )

        row[f"repeat_recall@{k}"] = (
            repeat_hits
            / len(repeat_truth)
            if repeat_truth
            else np.nan
        )

        row[f"new_recall@{k}"] = (
            new_hits
            / len(new_truth)
            if new_truth
            else np.nan
        )

    metric_rows.append(row)

candidate_metrics = pd.DataFrame(
    metric_rows
)

summary_rows = []

for k in RECALL_KS:
    for prefix in [
        "raw_recall",
        "catalog_recall",
        "repeat_recall",
        "new_recall",
    ]:
        summary_rows.append(
            {
                "metric": f"{prefix}@{k}",
                "value": candidate_metrics[
                    f"{prefix}@{k}"
                ].mean(),
            }
        )

summary_rows.extend(
    [
        {
            "metric": "catalog_coverage",
            "value": catalog_coverage,
        },
        {
            "metric": "users_evaluated",
            "value": len(candidate_metrics),
        },
        {
            "metric": "avg_candidates_per_user",
            "value": candidate_counts.mean(),
        },
    ]
)

candidate_summary = pd.DataFrame(
    summary_rows
)

display(candidate_summary)


,metric,value
0,raw_recall@50,0.649346
1,catalog_recall@50,0.681739
2,repeat_recall@50,0.976127
3,new_recall@50,0.280357
4,raw_recall@100,0.702332
5,catalog_recall@100,0.737503
6,repeat_recall@100,0.997263
7,new_recall@100,0.393263
8,raw_recall@200,0.741325
9,catalog_recall@200,0.778717


## 14. Candidate-source contribution


In [15]:
source_summary = (
    candidates["candidate_sources"]
    .value_counts()
    .rename_axis(
        "candidate_sources"
    )
    .reset_index(
        name="candidate_rows"
    )
)

source_summary["share"] = (
    source_summary["candidate_rows"]
    / source_summary[
        "candidate_rows"
    ].sum()
)

display(source_summary)


,candidate_sources,candidate_rows,share
0,svd,201510,0.503775
1,popularity|svd,107180,0.267950
2,popularity,61579,0.153947
3,history,12450,0.031125
4,history|svd,9958,0.024895
5,history|popularity|svd,7262,0.018155
6,history|popularity,61,0.000152


## 15. Attach base historical features to every generated candidate

This is the key architectural change from the previous Notebook 05.

We do **not** ask whether a generated pair already existed in Notebook 02.

Instead, every `(user, streamer)` candidate receives historical lookup features directly from raw history.

This creates the beginning of the dynamic inference feature layer.


In [16]:
pair_base = history_pairs.rename(
    columns={
        "interactions": "pair_history_interactions",
        "watch_minutes": "pair_history_watch_minutes",
        "last_time": "pair_last_history_time",
    }
)

candidate_features = (
    candidates
    .merge(
        pair_base,
        on=["user_id", "streamer"],
        how="left",
    )
    .merge(
        user_stats,
        on="user_id",
        how="left",
    )
    .merge(
        streamer_stats,
        on="streamer",
        how="left",
    )
)

candidate_features[
    "seen_before"
] = (
    candidate_features[
        "pair_history_interactions"
    ]
    .fillna(0)
    .gt(0)
    .astype("int8")
)

for col in [
    "pair_history_interactions",
    "pair_history_watch_minutes",
]:
    candidate_features[col] = (
        candidate_features[col]
        .fillna(0)
    )

candidate_features[
    "pair_history_share"
] = (
    candidate_features[
        "pair_history_interactions"
    ]
    / candidate_features[
        "user_history_interactions"
    ].replace(0, np.nan)
).fillna(0)

print(
    "Generated candidate rows:",
    f"{len(candidate_features):,}",
)
print(
    "Rows with base features:",
    f"{candidate_features['user_history_interactions'].notna().sum():,}",
)
print(
    "Seen-before candidate share:",
    f"{candidate_features['seen_before'].mean():.2%}",
)

display(
    candidate_features.head(10)
)


Generated candidate rows: 400,000
Rows with base features: 400,000
Seen-before candidate share: 7.45%


,user_id,streamer,retrieval_rank,retrieval_score,candidate_sources,history_rank,svd_rank,popularity_rank,pair_history_interactions,pair_history_watch_minutes,pair_last_history_time,user_history_interactions,user_history_watch_minutes,user_history_unique_streamers,user_last_history_time,streamer_history_interactions,streamer_history_watch_minutes,streamer_history_unique_viewers,streamer_avg_watch_minutes,seen_before,pair_history_share
0,27441,elded,1,0.029350,history|popularity|svd,1.0,40.0,126.0,7.0,80.0,4921.0,7,80.0,1,4921,2187,82290.0,685,37.626886,1,1.0
1,27441,tfue,2,0.025264,popularity|svd,NaN,1.0,2.0,0.0,0.0,NaN,7,80.0,1,4921,33636,883440.0,13261,26.264716,0,0.0
2,27441,ninja,3,0.025145,popularity|svd,NaN,2.0,1.0,0.0,0.0,NaN,7,80.0,1,4921,38701,968510.0,15581,25.025452,0,0.0
3,27441,elmiillor,4,0.019052,popularity|svd,NaN,3.0,113.0,0.0,0.0,NaN,7,80.0,1,4921,2403,80360.0,639,33.441531,0,0.0
4,27441,lvpes,5,0.018195,popularity|svd,NaN,4.0,154.0,0.0,0.0,NaN,7,80.0,1,4921,1834,70060.0,712,38.200654,0,0.0
5,27441,rubius,6,0.018015,popularity|svd,NaN,7.0,118.0,0.0,0.0,NaN,7,80.0,1,4921,2333,68630.0,1236,29.417060,0,0.0
6,27441,chocotaco,7,0.017003,popularity|svd,NaN,33.0,28.0,0.0,0.0,NaN,7,80.0,1,4921,5609,175140.0,1814,31.224817,0,0.0
7,27441,alexelcapo,8,0.016937,popularity|svd,NaN,9.0,165.0,0.0,0.0,NaN,7,80.0,1,4921,1758,62410.0,588,35.500569,0,0.0
8,27441,lirik,9,0.016679,popularity|svd,NaN,42.0,20.0,0.0,0.0,NaN,7,80.0,1,4921,7580,293440.0,2887,38.712401,0,0.0
9,27441,izakooo,10,0.015932,popularity|svd,NaN,19.0,108.0,0.0,0.0,NaN,7,80.0,1,4921,2439,70990.0,815,29.106191,0,0.0


## 16. Example recommendation funnel for one user


In [17]:
example_user = eval_users[0]

example = candidate_features[
    candidate_features[
        "user_id"
    ].eq(example_user)
].copy()

truth = future_truth.get(
    example_user,
    set(),
)

example[
    "is_future_positive"
] = example[
    "streamer"
].isin(truth)

print("User:", example_user)
print(
    "Historical streamers:",
    len(
        seen_lookup.get(
            example_user,
            set(),
        )
    ),
)
print(
    "Future-positive streamers:",
    len(truth),
)
print(
    "Future positives retrieved:",
    int(
        example[
            "is_future_positive"
        ].sum()
    ),
)

display(
    example[
        [
            "streamer",
            "retrieval_rank",
            "retrieval_score",
            "candidate_sources",
            "seen_before",
            "is_future_positive",
        ]
    ].head(30)
)


User: 27441
Historical streamers: 1
Future-positive streamers: 1
Future positives retrieved: 1


,streamer,retrieval_rank,retrieval_score,candidate_sources,seen_before,is_future_positive
0,elded,1,0.029350,history|popularity|svd,1,True
1,tfue,2,0.025264,popularity|svd,0,False
2,ninja,3,0.025145,popularity|svd,0,False
3,elmiillor,4,0.019052,popularity|svd,0,False
4,lvpes,5,0.018195,popularity|svd,0,False
5,rubius,6,0.018015,popularity|svd,0,False
6,chocotaco,7,0.017003,popularity|svd,0,False
7,alexelcapo,8,0.016937,popularity|svd,0,False
8,lirik,9,0.016679,popularity|svd,0,False
9,izakooo,10,0.015932,popularity|svd,0,False


## 17. Save Notebook 05 artifacts

Large raw event data are **not** duplicated.

We save only compact retrieval artifacts and aggregated lookup tables needed by later inference code.


In [18]:
candidate_path = (
    DATA_DIR
    / "twitch_generated_candidates_v2.csv.gz"
)

candidate_feature_path = (
    DATA_DIR
    / "twitch_candidate_base_features.csv.gz"
)

metrics_path = (
    RESULTS_DIR
    / "candidate_recall_by_user_v2.csv"
)

summary_path = (
    RESULTS_DIR
    / "candidate_summary_v2.csv"
)

source_path = (
    RESULTS_DIR
    / "candidate_source_summary_v2.csv"
)

candidates.to_csv(
    candidate_path,
    index=False,
    compression="gzip",
)

candidate_features.to_csv(
    candidate_feature_path,
    index=False,
    compression="gzip",
)

candidate_metrics.to_csv(
    metrics_path,
    index=False,
)

candidate_summary.to_csv(
    summary_path,
    index=False,
)

source_summary.to_csv(
    source_path,
    index=False,
)

# Compact historical lookup tables for later dynamic inference features.
history_pairs.to_csv(
    ARTIFACT_DIR
    / "retrieval_pair_history.csv.gz",
    index=False,
    compression="gzip",
)

user_stats.to_csv(
    ARTIFACT_DIR
    / "retrieval_user_stats.csv.gz",
    index=False,
    compression="gzip",
)

streamer_stats.to_csv(
    ARTIFACT_DIR
    / "retrieval_streamer_stats.csv.gz",
    index=False,
    compression="gzip",
)

joblib.dump(
    {
        "svd": svd,
        "user_ids": svd_user_ids,
        "streamers": svd_streamers,
        "user_factors": user_factors,
        "item_factors": item_factors,
        "user_to_idx": user_to_idx,
        "streamer_to_idx": streamer_to_idx,
        "source_weights": SOURCE_WEIGHTS,
        "rrf_k": RRF_K,
    },
    ARTIFACT_DIR
    / "hybrid_candidate_retriever.joblib",
)

metadata = {
    "cutoff_time": int(cutoff_time),
    "history_quantile": HISTORY_QUANTILE,
    "evaluation_users": int(len(eval_users)),
    "svd_users": int(len(svd_user_ids)),
    "svd_streamers": int(len(svd_streamers)),
    "svd_components": int(n_components),
    "max_candidates": int(MAX_CANDIDATES),
    "candidate_rows": int(len(candidates)),
    "catalog_coverage": float(catalog_coverage),
    "generation_seconds": float(generation_seconds),
    "source_weights": SOURCE_WEIGHTS,
    "candidate_metrics": {
        metric: (
            None
            if pd.isna(value)
            else float(value)
        )
        for metric, value
        in zip(
            candidate_summary[
                "metric"
            ],
            candidate_summary[
                "value"
            ],
        )
    },
}

with open(
    ARTIFACT_DIR
    / "candidate_generation_v2_metadata.json",
    "w",
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

print("Saved:")
print(candidate_path)
print(candidate_feature_path)
print(metrics_path)
print(summary_path)
print(source_path)
print(
    ARTIFACT_DIR
    / "hybrid_candidate_retriever.joblib"
)
print(
    ARTIFACT_DIR
    / "candidate_generation_v2_metadata.json"
)


Saved:
/Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/data/processed/twitch/twitch_generated_candidates_v2.csv.gz
/Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/data/processed/twitch/twitch_candidate_base_features.csv.gz
/Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/results/05_twitch_candidate_generation/candidate_recall_by_user_v2.csv
/Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/results/05_twitch_candidate_generation/candidate_summary_v2.csv
/Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/results/05_twitch_candidate_generation/candidate_source_summary_v2.csv
/Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/artifacts/twitch/hybrid_candidate_retriever.joblib
/Users/noopur/Desktop/resume_projects/Distributed ML Inference & Ranking/artifacts/twitch/candidate_generation_v2_metadata.json


# Notebook 05 v2 complete

The retrieval stage is now:

```text
raw historical interactions
        ↓
history + SVD collaborative retrieval + popularity
        ↓
reciprocal-rank fusion
        ↓
~200 candidates/user
        ↓
base historical feature lookup for every candidate
```

This resolves the previous dependency on the Notebook 02 candidate universe.

## What comes next

Notebook 06 should now evaluate the ranking models on a consistent population and prepare the **full dynamic 36-feature builder** that will be reused by the inference service.

The final serving path will be:

```text
user request
    ↓
candidate generator
    ↓
dynamic candidate features
    ↓
HGB / neural ranker
    ↓
Top-K
```
